# 🏆 Advanced Classification: GBDT Triad Ensemble

This notebook contains the interactive implementation of our production training pipeline. It trains LightGBM, XGBoost, and CatBoost models using 5-Fold Cross Validation, and applies Scipy Nelder-Mead optimization to find the best blending weights.

In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

# Ensure figures directory exists
os.makedirs('../figures', exist_ok=True)
print('Environment Initialized.')

## 1. Data Loading & Preprocessing

In [ ]:
train = pd.read_csv('../../../Pipeline/data/processed/train_processed.csv')

TARGET = 'health_condition'
X = train.drop(columns=[TARGET]).copy()
y_text = train[TARGET].copy()

le = LabelEncoder()
y = le.fit_transform(y_text)
num_classes = len(le.classes_)

print(f'Training Data Shape: {X.shape}')

## 2. Leak-Free OOF Target Encoding

In [ ]:
te_cols = ['stress_activity_combo', 'sleep_stress_combo', 'lifestyle_triad']
skf_te = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for c in te_cols:
    if c in X.columns:
        for cls in range(num_classes):
            col_name = f'{c}__te_cls{cls}'
            X[col_name] = 0.0
            y_cls = (y == cls).astype(float)
            
            for tr_idx, va_idx in skf_te.split(X, y):
                grp_means = X.iloc[tr_idx].groupby(c).apply(lambda d: y_cls[d.index].mean()).to_dict()
                global_mean = y_cls[tr_idx].mean()
                X.iloc[va_idx, X.columns.get_loc(col_name)] = X.iloc[va_idx][c].map(grp_means).fillna(global_mean)

# Format categoricals
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing').astype('category')

print(f'Train shape after Target Encoding: {X.shape}')

## 3. GBDT Triad Ensemble Training (Single Seed Demo)

In [ ]:
SEED = 42
num_folds = 5

oof_lgb = np.zeros((len(X), num_classes))
oof_xgb = np.zeros((len(X), num_classes))
oof_cat = np.zeros((len(X), num_classes))

lgb_params = {'n_estimators': 100, 'learning_rate': 0.05, 'num_leaves': 31, 'objective': 'multiclass', 'num_class': num_classes, 'verbose': -1, 'random_state': SEED}
xgb_params = {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 5, 'objective': 'multi:softprob', 'num_class': num_classes, 'eval_metric': 'mlogloss', 'enable_categorical': True, 'random_state': SEED}
cat_params = {'iterations': 100, 'learning_rate': 0.05, 'depth': 5, 'loss_function': 'MultiClass', 'verbose': 0, 'random_seed': SEED}

X_cat = X.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str).replace(['nan', 'None', 'NaN', 'N/A'], 'missing')

skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=SEED)

print('Training Models...')
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_va = y[train_idx], y[valid_idx]
    X_tr_cat, X_va_cat = X_cat.iloc[train_idx], X_cat.iloc[valid_idx]
    
    # LightGBM
    m1 = lgb.LGBMClassifier(**lgb_params)
    m1.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(20, verbose=False)])
    oof_lgb[valid_idx] = m1.predict_proba(X_va)
    
    # XGBoost
    m2 = xgb.XGBClassifier(**xgb_params)
    m2.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[valid_idx] = m2.predict_proba(X_va)
    
    # CatBoost
    m3 = CatBoostClassifier(**cat_params)
    m3.fit(X_tr_cat, y_tr, cat_features=cat_cols, eval_set=(X_va_cat, y_va), early_stopping_rounds=20, verbose=False)
    oof_cat[valid_idx] = m3.predict_proba(X_va_cat)

print(f'LGBM Balanced Accuracy: {balanced_accuracy_score(y, np.argmax(oof_lgb, axis=1)):.4f}')
print(f'XGB Balanced Accuracy: {balanced_accuracy_score(y, np.argmax(oof_xgb, axis=1)):.4f}')
print(f'CatBoost Balanced Accuracy: {balanced_accuracy_score(y, np.argmax(oof_cat, axis=1)):.4f}')

## 4. Scipy Optimization & Ensembling

In [ ]:
def model_blend_objective(weights):
    w1, w2, w3 = weights
    blend = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
    return -balanced_accuracy_score(y, np.argmax(blend, axis=1))

res_m = minimize(model_blend_objective, [0.33, 0.33, 0.33], method='Nelder-Mead', bounds=[(0.0, 1.0)]*3)
w1, w2, w3 = res_m.x
print(f'Optimized Blend Weights: [{w1:.4f}, {w2:.4f}, {w3:.4f}]')

blended_oof = w1 * oof_lgb + w2 * oof_xgb + w3 * oof_cat
master_cv_score = balanced_accuracy_score(y, np.argmax(blended_oof, axis=1))
print(f'🏆 FINAL BLENDED OOF BALANCED ACCURACY: {master_cv_score:.5f}')

## 5. Diagnostic Error Analysis & Figures

In [ ]:
opt_preds = np.argmax(blended_oof, axis=1)
cm = confusion_matrix(y, opt_preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Classification Confusion Matrix', fontsize=16)
plt.xlabel('Predicted')
plt.ylabel('Actual')

fig_path = '../figures/classification_confusion_matrix.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Figure saved to {fig_path}')

print('\nClassification Report:')
print(classification_report(y, opt_preds, target_names=le.classes_))

## 6. Saving Trained Models

In [ ]:

import joblib

os.makedirs('trained_models', exist_ok=True)

# Save the models from the first fold
joblib.dump(m1, 'trained_models/lgbm_model.joblib')
joblib.dump(m2, 'trained_models/xgb_model.joblib')
joblib.dump(m3, 'trained_models/cat_model.joblib')

print("✅ Models successfully saved to the 'trained_models/' directory!")
